In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageOps

# Check for RTX 3050
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

✅ Using device: cuda


In [2]:
# 0 is strictly reserved for the CTC Blank token
ALPHABET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ "
char_to_int = {char: i + 1 for i, char in enumerate(ALPHABET)}
int_to_char = {i + 1: char for i, char in enumerate(ALPHABET)}
NUM_CLASSES = len(ALPHABET) + 1 

print(f"✅ Vocabulary initialized with {NUM_CLASSES} classes.")

✅ Vocabulary initialized with 96 classes.


In [3]:
class WordDataset(Dataset):
    def __init__(self, txt_file, img_dir):
        self.img_dir = img_dir
        self.data = []
        with open(txt_file, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == 'ok':
                    filename = parts[0] + ".png" 
                    text = " ".join(parts[8:]) 
                    
                    # --- THE CRITICAL FIX ---
                    # Only keep the word if it contains characters we actually recognize
                    encoded_text = [char_to_int[c] for c in text if c in char_to_int]
                    
                    # 1. Must have at least one valid character
                    # 2. Must be short enough for our CNN output width (max 16 chars)
                    if len(encoded_text) > 0 and len(encoded_text) <= 16:
                        self.data.append((filename, encoded_text))
                    # ------------------------
                    
        self.transform = transforms.ToTensor()

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        filename, encoded_text = self.data[idx]

        word_id = filename.replace(".png", "")
        parts = word_id.split('-')

        folder1 = parts[0]
        folder2 = parts[0] + '-' + parts[1]

        img_path = os.path.join(
            self.img_dir,
            folder1,
            folder2,
            filename
        )

        try:
            im = Image.open(img_path).convert('L')
        except Exception:
            print("Bad image:", img_path)
            return self.__getitem__((idx + 1) % len(self.data))

        new_w = int(im.size[0] * (32 / im.size[1]))
        im = im.resize((new_w, 32), Image.LANCZOS)

        if new_w < 128:
            im = ImageOps.expand(im, (0, 0, 128 - new_w, 0), fill=255)
        else:
            im = im.crop((0, 0, 128, 32))

            return self.transform(im), torch.tensor(encoded_text, dtype=torch.long)
        
        def collate_fn(batch):
            images, labels = zip(*batch)

            images = torch.stack(images)

            target_lengths = torch.tensor(
            [len(label) for label in labels],
            dtype=torch.long
        )

            targets = torch.cat(labels)

            return images, targets, target_lengths

In [4]:
class HandwrittenWordReader(nn.Module):
    def __init__(self, num_chars, hidden_size=256, num_layers=2):
        super(HandwrittenWordReader, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2, 2), (2, 1), (0, 1)) 
        )
        self.rnn = nn.LSTM(256 * 4, hidden_size, num_layers, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, num_chars)

    def forward(self, x):
        conv_out = self.cnn(x)
        b, c, h, w = conv_out.size()
        conv_out = conv_out.view(b, c * h, w).permute(0, 2, 1) 
        rnn_out, _ = self.rnn(conv_out)
        return F.log_softmax(self.fc(rnn_out), dim=2).permute(1, 0, 2)

model = HandwrittenWordReader(num_chars=NUM_CLASSES).to(device)

In [7]:
# Paths for your local AryanPC setup
import torch

def collate_fn(batch):
    # Unpack the batch (assumes your dataset returns (image, target))
    images, targets = zip(*batch)
    
    # 1. Batch the images
    # Assumes images are already resized to the same shape in your Dataset
    images = torch.stack(images, 0)
    
    # 2. Get the length of each target
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    
    # 3. Concatenate targets into a single 1D tensor
    # If your dataset returns python lists instead of tensors, we convert them first
    if not isinstance(targets[0], torch.Tensor):
        targets = [torch.tensor(t, dtype=torch.long) for t in targets]
        
    targets = torch.cat(targets, 0)
    
    return images, targets, target_lengths

word_train_ds = WordDataset('iam_data/words_new.txt', 'iam_data/iam_words/words')
train_loader = DataLoader(word_train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CTCLoss(blank=0, zero_infinity=True) 

for epoch in range(50):
    model.train()
    running_loss = 0.0
    for images, targets, target_lengths in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        preds = model(images)
        input_lengths = torch.full((images.size(0),), preds.size(0), dtype=torch.long)
        loss = criterion(preds, targets, input_lengths, target_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) # Circuit breaker
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/50 | Loss: {running_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), 'crnn_word_reader_final.pth')

TypeError: 'NoneType' object is not iterable